# TravelMind Coding Exercise 1: Beginner
### Give after ~30% of Day 6 (single agent, tools, chaining, routing)

You are on shift at TravelMind. The senior team is prepping for the weekend storm and left you the calm queue. Your job: wire the first working versions and prove they run.

**What is inside (work top to bottom):**
- Setup you run once (model, mock airline tools, a cost meter).
- Nine tasks that mix: MCQ, choose-an-option, fill-the-blank, a table, a flowchart to complete, spot-the-error, debug-and-fix, and implement-from-a-flow with a predict-the-output check.

**Rules of the room:**
- Anchor booking: PNR `JX48Q2`, surname `Rao`, Gold tier, `BLR-DEL` cancelled by the airline.
- Run the setup cells first. Then each task has a cell you edit. Placeholders are `____` or `# TODO`.
- The one question under every task: who controls the path, you or the model?

## How to run
- **VS Code:** open this notebook, pick a Python 3.10+ kernel, make sure boto3 has AWS credentials with Bedrock access to Claude Haiku 4.5 in `us-east-1`, run top to bottom.
- **Colab:** upload, run the install cell, set `os.environ["AWS_ACCESS_KEY_ID"]=...` and `os.environ["AWS_REGION"]="us-east-1"`, run top to bottom.

In [ ]:
%pip install -q strands-agents strands-agents-tools matplotlib

In [ ]:
import os, time, json, contextlib
from strands import Agent, tool
from strands.models import BedrockModel

REGION   = os.environ.get("AWS_REGION", "us-east-1")
HAIKU_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # us. => cross-region inference profile
haiku    = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0.3)
print("Model ready:", HAIKU_ID)

In [ ]:
# Mock airline operations (deterministic fake data; no reservation system needed).
_PNRS = {"JX48Q2": {"surname": "Rao", "passenger_id": "P-100294", "loyalty_tier": "Gold",
    "segments": [{"flight": "6E-317", "from": "BLR", "to": "DEL", "dep": "2026-07-05T08:10",
                  "status": "CANCELLED", "fare_basis": "TA21R"},
                 {"flight": "6E-512", "from": "DEL", "to": "BOM", "dep": "2026-07-05T13:40",
                  "status": "ON TIME", "fare_basis": "TA21R"}],
    "ancillaries": {"seat": "14C (paid)", "bags": 1}}}
_FARE_RULES = {"TA21R": {"refundable": False, "change_fee": 3000, "currency": "INR",
    "notes": "Non-refundable on voluntary change. Airline-caused (involuntary) changes waive fees."}}


@tool
def get_pnr(record_locator: str, surname: str) -> str:
    """Look up a booking by record locator and surname (identity check).

    Args:
        record_locator: 6-character PNR, e.g. 'JX48Q2'.
        surname: Passenger surname for verification.
    Returns:
        JSON string of the booking, or an error if not found / surname mismatch.
    """
    rec = _PNRS.get(record_locator.upper())
    if not rec or rec["surname"].lower() != surname.lower():
        return json.dumps({"error": "PNR not found or surname mismatch"})
    return json.dumps(rec)


@tool
def get_fare_rules(fare_basis: str) -> str:
    """Return fare rules (refundability, change fee) for a fare basis code.

    Args:
        fare_basis: Fare basis code, e.g. 'TA21R'.
    Returns:
        JSON string of fare rules.
    """
    return json.dumps(_FARE_RULES.get(fare_basis.upper(), {"error": "unknown fare basis"}))


@tool
def search_reaccommodation(origin: str, destination: str, after: str) -> str:
    """Find alternative flights after a disruption.

    Args:
        origin: Origin airport code, e.g. 'BLR'.
        destination: Destination airport code, e.g. 'BOM'.
        after: ISO datetime; only flights departing after this are returned.
    Returns:
        JSON string list of candidate flights.
    """
    return json.dumps([{"flight": "6E-333", "from": origin, "to": destination, "dep": "2026-07-05T11:20", "seats": 4},
                       {"flight": "AI-809", "from": origin, "to": destination, "dep": "2026-07-05T15:05", "seats": 9}])


@tool
def get_loyalty(passenger_id: str) -> str:
    """Return loyalty tier and benefits for a passenger.

    Args:
        passenger_id: Internal passenger id, e.g. 'P-100294'.
    Returns:
        JSON string of tier and benefits.
    """
    return json.dumps({"passenger_id": passenger_id, "tier": "Gold",
                       "benefits": ["priority rebooking", "waived change fee on same-day", "lounge access"]})


@tool
def check_refund_eligibility(record_locator: str, reason: str) -> str:
    """Assess refund eligibility from fare rules and the stated reason.

    Args:
        record_locator: PNR.
        reason: Free-text reason, e.g. 'flight cancelled by airline'.
    Returns:
        JSON string with an eligibility signal and the controlling rule.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"error": "PNR not found"})
    rules = _FARE_RULES.get(rec["segments"][0]["fare_basis"], {})
    airline_caused = any(k in reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    return json.dumps({"eligible": airline_caused or rules.get("refundable", False),
                       "airline_caused": airline_caused, "rule": rules.get("notes", "")})

print("Tools ready.")

In [ ]:
# Tiny cost meter. Illustrative Haiku prices (verify before quoting): input $1.00, output $5.00 per 1M tokens.
PRICES = {"haiku": (1.00, 5.00)}
_LEDGER, RESULTS = [], {}

def usage_of(result):
    u = getattr(getattr(result, "metrics", None), "accumulated_usage", None) or {}
    return {"input": u.get("inputTokens", 0), "output": u.get("outputTokens", 0)}

def metered(agent, prompt, tier="haiku"):
    res = agent(prompt); _LEDGER.append((tier, usage_of(res))); return str(res)

@contextlib.contextmanager
def meter(label):
    _LEDGER.clear(); t0 = time.time()
    try:
        yield
    finally:
        lat = time.time() - t0; cost = tin = tout = 0
        for tier, u in _LEDGER:
            pin, pout = PRICES[tier]
            cost += u["input"]/1e6*pin + u["output"]/1e6*pout; tin += u["input"]; tout += u["output"]
        RESULTS[label] = {"latency_s": round(lat, 2), "tokens": tin + tout,
                          "calls": len(_LEDGER), "cost_usd": round(cost, 6)}
        print(f"[{label}] calls={len(_LEDGER)} latency={round(lat,2)}s tokens={tin+tout} cost=${round(cost,6):.6f}")

print("Meter ready.")

## Task 1: MCQ, pick the pattern

Priya drops three tickets:
- **T-A:** "What's the status of JX48Q2?"
- **T-B:** every refund notice runs the same three steps in the same order, always.
- **T-C:** messages split cleanly into status, change, or refund.

Set each answer in the cell. One letter each.

In [ ]:
# a) augmented agent   b) prompt chaining   c) routing   d) swarm
ANSWER_TA = "?"   # T-A: a single status lookup
ANSWER_TB = "?"   # T-B: fixed three-step order
ANSWER_TC = "?"   # T-C: clean categories, one specialist each
print(ANSWER_TA, ANSWER_TB, ANSWER_TC)

## Task 2: Fill the blank, build the augmented agent

Complete the smallest useful TravelMind: one agent, one job, one tool.
- Give it a clear `name`.
- Give it the single tool that looks up a booking.

In [ ]:
travelmind = Agent(
    model=haiku,
    name="____",                       # TODO: any clear name
    system_prompt="Report flight status for a PNR. Verify identity first.",
    tools=[____],                      # TODO: which tool looks up a booking?
)

with meter("t2_augmented"):
    print(metered(travelmind, "Hi, this is Rao. Status of booking JX48Q2?"))

## Task 3: Debug, the model will not call

Run the cell. It fails on the first call with `on-demand throughput isn't supported`. Fix the one thing that is wrong, then re-run.

In [ ]:
broken_model = BedrockModel(
    model_id="anthropic.claude-haiku-4-5-20251001-v1:0",   # something is missing here
    region_name="us-east-1",
    temperature=0.3,
)
test_agent = Agent(model=broken_model, name="test", system_prompt="Reply with one word: ok.")
print(test_agent("ping"))   # run, watch it fail, then fix the line above

## Task 4: Spot the errors (no running needed)

This tool is meant to be callable by an agent. It has **two** problems that will make the agent use it badly. Read it and fill in the two blanks below.

```python
@tool
def seat_map(pnr):
    return {"seat": "14C", "status": "paid"}
```

- Problem 1 (the agent cannot tell what this tool is for): ________
- Problem 2 (the return type an agent tool should hand back): ________

In [ ]:
# Record your two findings as short strings.
ERROR_1 = "____"   # TODO
ERROR_2 = "____"   # TODO
print(ERROR_1); print(ERROR_2)

## Task 5: Implement from the flow, prompt chaining

Build a fixed three-step pipeline. You wrote the order, so this is a workflow, not an agent deciding anything.

```mermaid
flowchart LR
    In([Request]) --> S1[extractor: intent + entities as JSON]
    S1 --> S2[resolver: gather facts with tools]
    S2 --> S3[writer: warm customer reply]
    S3 --> Out([Reply])
```

Instructions:
1. `extractor`: no tools, outputs intent and entities as JSON.
2. `resolver`: tools are `get_pnr`, `get_fare_rules`, `search_reaccommodation`; summarizes the options.
3. `writer`: no tools, turns the resolver output into a reply.
4. Wire them so each output feeds the next, inside the `meter` block.

In [ ]:
extractor = Agent(model=haiku, name="extractor", system_prompt="____")                 # TODO
resolver  = Agent(model=haiku, name="resolver",  system_prompt="____", tools=[____])     # TODO
writer    = Agent(model=haiku, name="writer",    system_prompt="____")                   # TODO

q = "This is Rao, PNR JX48Q2. My BLR-DEL flight was cancelled. What are my options to reach BOM today?"

with meter("t5_chain"):
    intent = ____                    # TODO: run extractor on q
    facts  = ____                    # TODO: run resolver on the intent plus q
    reply  = ____                    # TODO: run writer on facts
    print(reply)

## Task 6: Predict, then check

A cheap classifier returns exactly one of: `status`, `change`, `refund`.

Before running the next task's router, write your prediction for this message:

> "Rao here, JX48Q2. My flight was cancelled by the airline. Can I get a refund?"

- Predicted label: ________
- Predicted specialist that runs: ________

In [ ]:
PREDICTED_LABEL = "____"        # TODO: status / change / refund
PREDICTED_SPECIALIST = "____"   # TODO: which *_agent runs
print("I predict:", PREDICTED_LABEL, "->", PREDICTED_SPECIALIST)

## Task 7: Fill the table

Each specialist should carry only the tools it needs. Fill the tool list for each.

| Specialist | Job | Tools it needs |
|---|---|---|
| status_agent | report segment status | ________ |
| change_agent | find a new flight, check fees | ________ |
| refund_agent | decide refund eligibility | ________ |

Then record your answers as Python lists below (use the real tool names).

In [ ]:
STATUS_TOOLS = [____]   # TODO
CHANGE_TOOLS = [____]   # TODO
REFUND_TOOLS = [____]   # TODO
print("status:", STATUS_TOOLS); print("change:", CHANGE_TOOLS); print("refund:", REFUND_TOOLS)

## Task 8: Complete the flowchart, then implement routing

Fill the three blank branches, then implement the router.

```mermaid
flowchart TD
    In([Message]) --> C{cheap classifier}
    C -->|... fill 1| A1[status_agent]
    C -->|... fill 2| A2[change_agent]
    C -->|... fill 3| A3[refund_agent]
```

Implement `route`: classify the message, pick the matching specialist, run it, return the label and reply. The heavy work must only run on the branch taken.

In [ ]:
CATEGORIES = ["status", "change", "refund"]
classifier = Agent(model=haiku, name="classifier",
    system_prompt="Classify the message into exactly one of " + str(CATEGORIES) + ". Output only the label.")

status_agent = Agent(model=haiku, name="status_agent",
    system_prompt="Handle flight status. Verify identity, then report each segment.", tools=[get_pnr])
change_agent = Agent(model=haiku, name="change_agent",
    system_prompt="Handle changes. Check fare rules and find re-accommodation.",
    tools=[get_pnr, get_fare_rules, search_reaccommodation])
refund_agent = Agent(model=haiku, name="refund_agent",
    system_prompt="Handle refunds. Always check eligibility before any promise.",
    tools=[get_pnr, check_refund_eligibility])
SPECIALISTS = {"status": status_agent, "change": change_agent, "refund": refund_agent}

def route(msg):
    label = ____                                   # TODO: classify msg, keep just the label word
    label = label if label in SPECIALISTS else "status"
    reply = ____                                   # TODO: run the chosen specialist on msg
    return label, reply

with meter("t8_route"):
    lbl, rep = route("Rao here, JX48Q2, flight cancelled by the airline. Can I get a refund?")
    print("routed ->", lbl); print(rep)

## Task 9: Choose the design, and say why

For the status lookup in Task 2, two designs are proposed:
- **Design 1:** one `Agent` with `get_pnr`.
- **Design 2:** a three-agent `Swarm` (lookup, backup, reviewer) with handoffs.

Pick one, and write the one thing the loser costs you without buying it back.

In [ ]:
CHOICE = "____"     # TODO: "Design 1" or "Design 2"
REASON = "____"     # TODO: one line, what the loser costs
print(CHOICE, "-", REASON)

## Done
You built v1 (augmented), v2 (chaining), and v3 (routing), and measured each. Every one kept you in control of the path. That is the floor of the ladder. The next notebook hands part of the path to the model.